In [1]:
import time
import csv
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
from torchvision.models import resnet18, ResNet18_Weights


def count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def count_total_params(model):
    return sum(p.numel() for p in model.parameters())


def get_peak_memory():
    if torch.cuda.is_available():
        return torch.cuda.max_memory_allocated() / 1024**2  # MB
    return 0.0


def build_resnet18_bitfit(num_classes=101):
    """
    BitFit for ResNet18:
    - load ImageNet pretrained weights
    - replace classifier for Food101
    - freeze everything
    - unfreeze only:
        1) all bias parameters
        2) final classifier head
    """
    model = models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
    model.fc = nn.Linear(model.fc.in_features, num_classes)

    # Freeze everything first
    for param in model.parameters():
        param.requires_grad = False

    # Unfreeze all bias parameters
    for name, param in model.named_parameters():
        if "bias" in name:
            param.requires_grad = True

    # Unfreeze full classifier head
    for param in model.fc.parameters():
        param.requires_grad = True

    return model


In [2]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(loader)
    acc = correct / total
    return avg_loss, acc

In [3]:
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / len(loader)
    acc = correct / total
    return avg_loss, acc


In [4]:
os.makedirs("results", exist_ok=True)
os.makedirs("models", exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

csv_file = "results/ResNet18_BitFit_metrics.csv"

if not os.path.exists(csv_file):
    with open(csv_file, mode="w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            "method", "epoch",
            "train_loss", "train_acc",
            "val_loss", "val_acc",
            "epoch_time_sec",
            "peak_memory_mb",
            "trainable_params",
            "total_params"
        ])

# ImageNet-1K pretrained weights
weights = ResNet18_Weights.IMAGENET1K_V1

# Use pretrained preprocessing
transform = weights.transforms()

# Load Food101
train_dataset_full = datasets.Food101(
    root="./data",
    split="train",
    download=True,
    transform=transform
)

test_dataset = datasets.Food101(
    root="./data",
    split="test",
    download=True,
    transform=transform
)

# Split training into train + validation
train_size = int(0.9 * len(train_dataset_full))
val_size = len(train_dataset_full) - train_size

train_dataset, val_dataset = random_split(train_dataset_full, [train_size, val_size])

# DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=4,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=4,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=4,
    pin_memory=torch.cuda.is_available()
)

# Build BitFit model
model = build_resnet18_bitfit(num_classes=101)
model = model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=0.001
)

num_epochs = 8
best_val_acc = 0.0

trainable_params = count_trainable_params(model)
total_params = count_total_params(model)

print(f"Total params: {total_params:,}")
print(f"Trainable params: {trainable_params:,}")
print(f"Trainable percentage: {100 * trainable_params / total_params:.4f}%")

print("\nTrainable parameter names:")
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name, param.shape)

for epoch in range(num_epochs):
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    start_time = time.time()

    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)

    epoch_time = time.time() - start_time
    peak_memory = get_peak_memory()

    print(f"Epoch {epoch + 1}/{num_epochs}")
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Val Loss:   {val_loss:.4f}, Val Acc:   {val_acc:.4f}")
    print(f"Time: {epoch_time:.2f}s | Peak Mem: {peak_memory:.2f} MB")
    print("-" * 40)

    # Save metrics to CSV
    with open(csv_file, mode="a", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            "resnet18_bitfit",
            epoch + 1,
            train_loss,
            train_acc,
            val_loss,
            val_acc,
            epoch_time,
            peak_memory,
            trainable_params,
            total_params
        ])

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc

        model_path = "models/ResNet18_BitFit_best_21_april.pth"
        torch.save({
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_acc": val_acc,
            "epoch": epoch
        }, model_path)

        print("Best model saved!")

# Final test evaluation using best loaded checkpoint if desired
print(f"\nBest validation accuracy: {best_val_acc:.4f}")

test_loss, test_acc = evaluate(model, test_loader, criterion, device)
print(f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}")

Using device: cuda
Total params: 11,228,325
Trainable params: 56,613
Trainable percentage: 0.5042%

Trainable parameter names:
bn1.bias torch.Size([64])
layer1.0.bn1.bias torch.Size([64])
layer1.0.bn2.bias torch.Size([64])
layer1.1.bn1.bias torch.Size([64])
layer1.1.bn2.bias torch.Size([64])
layer2.0.bn1.bias torch.Size([128])
layer2.0.bn2.bias torch.Size([128])
layer2.0.downsample.1.bias torch.Size([128])
layer2.1.bn1.bias torch.Size([128])
layer2.1.bn2.bias torch.Size([128])
layer3.0.bn1.bias torch.Size([256])
layer3.0.bn2.bias torch.Size([256])
layer3.0.downsample.1.bias torch.Size([256])
layer3.1.bn1.bias torch.Size([256])
layer3.1.bn2.bias torch.Size([256])
layer4.0.bn1.bias torch.Size([512])
layer4.0.bn2.bias torch.Size([512])
layer4.0.downsample.1.bias torch.Size([512])
layer4.1.bn1.bias torch.Size([512])
layer4.1.bn2.bias torch.Size([512])
fc.weight torch.Size([101, 512])
fc.bias torch.Size([101])
Epoch 1/8
Train Loss: 2.1928, Train Acc: 0.4665
Val Loss:   1.6894, Val Acc:   0.